# Policy iteration on the 3x3 grid world

1. **Initialization** — pick $V(s)$ and $\pi(s)$ arbitrarily.
2. **Policy evaluation** — sweep the Bellman expectation equation until $\Delta < \theta$.
3. **Policy improvement** — act greedily w.r.t. $V$; if nothing changed, stop.

States are numbered left to right, top to bottom ($s_1,\ldots,s_9$), as before.

In [46]:
import numpy as np

## The environment

|        | col 1 | col 2 | col 3 |
|--------|-------|-------|-------|
| row 1  | $s_1$ white | $s_2$ white | $s_3$ white |
| row 2  | $s_4$ white | $s_5$ white | $s_6$ **orange** |
| row 3  | $s_7$ **orange** | $s_8$ white | $s_9$ **blue (target)** |

* 5 actions: `0 = up`, `1 = right`, `2 = down`, `3 = left`, `4 = stay`.
* Transitions are deterministic.
* The reward depends on the cell you *enter*: $+1$ for the blue cell, $-1$ for an orange
  cell, $0$ for a white cell, and $-1$ if the action would take you off the grid
  (in that case you also stay where you are).
* Continuing task: $s_9$ is **not** absorbing, staying there keeps paying $+1$.

In [47]:
# Parameters
gamma         = 0.9      # discount factor
theta         = 1e-6     # accuracy threshold for policy evaluation

n_states      = 9
n_actions     = 5

action_names  = ["up", "right", "down", "left", "stay"]
action_arrows = ["^", ">", "v", "<", "o"]

## The model

Since everything is deterministic, the model
$p(s', r | s, a)$ collapses into two $9 \times 5$ tables:

$$\texttt{NEXT}[s, a] = s' \qquad \texttt{REWARD}[s, a] = r.$$

Rows are states $s_1,\ldots,s_9$, columns are the five actions. Python indexes
from 0, so state $s_i$ lives at row `i-1` (e.g. $s_9$ is row `8`).

In [48]:
# Next state s' reached from state s (row) by action a (column).
NEXT = np.array([
#   up  right  down  left  stay
    [0,   1,    3,    0,    0],   # s1
    [1,   2,    4,    0,    1],   # s2
    [2,   2,    5,    1,    2],   # s3
    [0,   4,    6,    3,    3],   # s4
    [1,   5,    7,    3,    4],   # s5
    [2,   5,    8,    4,    5],   # s6  (orange)
    [3,   7,    6,    6,    6],   # s7  (orange)
    [4,   8,    7,    6,    7],   # s8
    [5,   8,    8,    7,    8],   # s9  (blue target)
])

# Reward collected on that transition.
REWARD = np.array([
#   up  right  down  left  stay
    [-1,   0,    0,   -1,    0],  # s1
    [-1,   0,    0,    0,    0],  # s2
    [-1,  -1,   -1,    0,    0],  # s3
    [ 0,   0,   -1,   -1,    0],  # s4
    [ 0,  -1,    0,    0,    0],  # s5
    [ 0,  -1,    1,    0,   -1],  # s6  (orange: staying costs -1)
    [ 0,   0,   -1,   -1,   -1],  # s7  (orange)
    [ 0,   1,   -1,   -1,    0],  # s8
    [-1,  -1,   -1,    0,    1],  # s9  (blue: staying pays +1)
], dtype=float)

Two small helpers to print a value function and a policy as a 3x3 grid.

In [49]:
def show_values(V, label="value function"):
    print(label + ":")
    print(np.array2string(V.reshape(3, 3), precision=2, floatmode="fixed"))

def show_policy(pi, label="policy"):
    print(label + ":")
    arrows = np.array([action_arrows[a] for a in pi]).reshape(3, 3)
    for row in arrows:
        print("  " + "  ".join(row))

## Initialization

$V(s) \in \mathbb{R}$ and $\pi(s) \in \mathcal{A}(s)$ arbitrarily. We take $V \equiv 0$ and policy "always stay".

In [50]:
V  = np.zeros(n_states)
pi = np.full(n_states, 4)   # action 4 = stay, in every state

show_values(V, "initial V")
show_policy(pi, "initial policy")

initial V:
[[0.00 0.00 0.00]
 [0.00 0.00 0.00]
 [0.00 0.00 0.00]]
initial policy:
  o  o  o
  o  o  o
  o  o  o


## Policy evaluation

$$V(s) \leftarrow \sum_{s', r} p(s', r \mid s, \pi(s)) \left[ r + \gamma V(s') \right]
 \;=\; \texttt{REWARD}[s, \pi(s)] + \gamma\, V\big(\texttt{NEXT}[s, \pi(s)]\big),$$

the sum having a single term because the environment is deterministic. Sweeps are done
**in place** (the new $V(s)$ is used immediately by the following states), and we stop
when the largest change in a whole sweep is below $\theta$.

In [51]:
def policy_evaluation(V, pi):
    while True:
        delta = 0.0
        for s in range(n_states):
            v = V[s]
            a = pi[s]
            V[s] = REWARD[s, a] + gamma * V[NEXT[s, a]]
            delta = max(delta, abs(v - V[s]))
        if delta < theta:
            break
    return V

## Policy improvement

$$\pi(s) \leftarrow \arg\max_a \; \texttt{REWARD}[s, a] + \gamma\, V\big(\texttt{NEXT}[s, a]\big).$$

The flag `policy_stable` tells us whether any state changed its action. If none did, the
policy is greedy with respect to its own value function, which is exactly the Bellman
optimality equation, and we are done.

In [52]:
def policy_improvement(V, pi):
    policy_stable = True
    for s in range(n_states):
        old_action = pi[s]
        # action values q(s,a) for the five actions
        q = np.array([REWARD[s, a] + gamma * V[NEXT[s, a]] for a in range(n_actions)])
        pi[s] = np.argmax(q)        # ties are broken by the lowest action index
        if old_action != pi[s]:
            policy_stable = False
    return pi, policy_stable

## The policy iteration loop

Evaluate, improve, repeat. Note that evaluation restarts from the value function of the
*previous* policy, not from zeros, which is why it converges so fast.

In [53]:
V  = np.zeros(n_states)
pi = np.full(n_states, 4)

for k in range(100):
    V = policy_evaluation(V, pi)
    pi, policy_stable = policy_improvement(V, pi)

    print(f"--- iteration {k}")
    show_values(V)
    show_policy(pi)
    print()

    if policy_stable:
        print(f"policy stable after {k + 1} iterations")
        break

--- iteration 0
value function:
[[  0.00   0.00   0.00]
 [  0.00   0.00 -10.00]
 [-10.00   0.00  10.00]]
policy:
  >  >  <
  ^  ^  v
  ^  >  o

--- iteration 1
value function:
[[ 0.00  0.00  0.00]
 [ 0.00  0.00 10.00]
 [ 0.00 10.00 10.00]]
policy:
  >  >  v
  ^  v  v
  >  >  o

--- iteration 2
value function:
[[ 6.48  7.20  8.00]
 [ 5.83  9.00 10.00]
 [ 9.00 10.00 10.00]]
policy:
  >  v  v
  >  v  v
  >  >  o

--- iteration 3
value function:
[[ 7.29  8.10  8.00]
 [ 8.10  9.00 10.00]
 [ 9.00 10.00 10.00]]
policy:
  >  v  v
  >  v  v
  >  >  o

policy stable after 4 iterations


## Result

`V` is now $v_*$ and `pi` is an optimal policy $\pi_*$.

In [54]:
show_values(V, "optimal value function v*")
print()
show_policy(pi, "optimal policy pi*")

optimal value function v*:
[[ 7.29  8.10  8.00]
 [ 8.10  9.00 10.00]
 [ 9.00 10.00 10.00]]

optimal policy pi*:
  >  v  v
  >  v  v
  >  >  o
